# Demo End-to-End: Nén Tín Hiệu PPG + ACC (1D-CNN Autoencoder vs. DCT-II)

> **Môn học / Đề tài:** Nghiên cứu và So sánh Thuật toán Nén Tín hiệu Cảm biến Đa kênh trên Thiết bị Đeo IoT (PPG-DaLiA Dataset)  
> **Tác giả:** C1_AE_DCT Team  

---

## 1. Giới thiệu Đề tài

Bài toán đặt ra là tối ưu hóa việc truyền tải dữ liệu cảm biến đa kênh liên tục từ thiết bị đeo Empatica E4 trong bộ dữ liệu **PPG-DaLiA**:
- **Wrist PPG:** Tín hiệu Quang thể tích đồ tần số **64 Hz** (1 kênh).
- **Wrist ACC (x, y, z):** Tín hiệu Gia tốc kế tần số **32 Hz** (3 kênh).

### Luồng xử lý dữ liệu:
1. **Resampling:** ACC 32 Hz được đồng bộ lên **64 Hz** bằng bộ lọc Polyphase (`scipy.signal.resample_poly`).
2. **Windowing & Normalization:** Cắt cửa sổ $T = 8\text{s}$ (**512 mẫu** $\times$ 4 kênh = 2,048 mẫu vật lý). Chuẩn hóa Z-score với thống kê chỉ tính từ tập Train từng Fold.
3. **So sánh Nén:**
   - **1D-CNN Autoencoder (AE):** Nén không gian 4 kênh thành không gian ẩn $M = 32 \times d_b$ (với $d_b = 8 \implies M = 256$).
   - **Discrete Cosine Transform (DCT-II):** Khớp ngân sách chiều (**Equal-Dimension:** $K=M=256$) và ngân sách byte (**Equal-Byte:** $K=170$, $B_{\text{DCT}} = B_{\text{AE}} = 1040\text{ Bytes}$).
4. **Mục tiêu Demo:** Kiểm chứng minh họa end-to-end pipeline tái tạo tín hiệu và các chỉ số méo dạng (PRD, PRDN, RMSE) trên dữ liệu PPG-DaLiA thực tế.

---
## 2. Import Module và Kiểm tra Môi trường Chạy

In [1]:
# ==============================================================================
# SECTION 2 — IMPORT VÀ KIỂM TRA MÔI TRƯỜNG
# ==============================================================================
import sys
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

# 1. Path handling: Tự động xác định PROJECT_ROOT linh hoạt
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2. Import các module dự án từ src/
import src.dataset as dataset
import src.resample as resample
import src.normalize as normalize
import src.windowing as windowing
import src.model as model_module
import src.baseline_dct as baseline_dct
import src.codec as codec
import src.evaluate as evaluate
import src.metrics as metrics

# 3. Kiểm tra thiết bị tính toán
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[INFO] PROJECT_ROOT : {PROJECT_ROOT}")
print(f"[INFO] Python       : {platform.python_version()}")
print(f"[INFO] PyTorch      : {torch.__version__}")
print(f"[INFO] Device       : {device}")

[INFO] PROJECT_ROOT : D:\UTE\AIForIOT\project_Cuoiki_iot\C1_AE_DCT
[INFO] Python       : 3.14.5
[INFO] PyTorch      : 2.14.0+cpu
[INFO] Device       : cpu


---
## 3. Kiểm tra Bộ dữ liệu Thực tế (PPG-DaLiA)

Sử dụng module `src.dataset` để quét thư mục chứa 15 subjects ($S1 \dots S15$) và nạp thông tin subject $S1$.

In [2]:
# ==============================================================================
# SECTION 3 — KIỂM TRA DATASET
# ==============================================================================
# Tìm root dataset PPG-DaLiA
dataset_root = dataset.find_dataset_root(PROJECT_ROOT / "data" / "raw")
print(f"[DATASET] Dataset Root: {dataset_root}")

# Kiểm tra danh sách 15 subjects
subjects = [f"S{i}" for i in range(1, 16)]
print(f"[DATASET] Danh sach subjects: {subjects}")

# Nạp dữ liệu pickle thực tế cho Subject S1
s1_data = dataset.load_subject_pickle("S1", dataset_dir=dataset_root)
ppg_raw, acc_raw, meta = dataset.extract_wrist_signals(s1_data)

ppg_duration = len(ppg_raw) / 64.0
acc_duration = len(acc_raw) / 32.0

has_nan = np.isnan(ppg_raw).any() or np.isnan(acc_raw).any()
has_inf = np.isinf(ppg_raw).any() or np.isinf(acc_raw).any()

print("\n[DATASET] Thong so Subject S1 Raw Signals:")
print(f"  - Wrist PPG (64 Hz) shape : {ppg_raw.shape}, Thoi luong: {ppg_duration:.2f} s")
print(f"  - Wrist ACC (32 Hz) shape : {acc_raw.shape}, Thoi luong: {acc_duration:.2f} s")
print(f"  - Tin hieu chua NaN       : {has_nan}")
print(f"  - Tin hieu chua Inf       : {has_inf}")

[DATASET] Dataset Root: D:\UTE\AIForIOT\project_Cuoiki_iot\ppg+dalia\data\PPG_FieldStudy
[DATASET] Danh sach subjects: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15']



[DATASET] Thong so Subject S1 Raw Signals:
  - Wrist PPG (64 Hz) shape : (589568,), Thoi luong: 9212.00 s
  - Wrist ACC (32 Hz) shape : (294784, 3), Thoi luong: 9212.00 s
  - Tin hieu chua NaN       : False
  - Tin hieu chua Inf       : False


---
## 4. Hiển thị Tín hiệu Gốc Trước Resample

Trực quan hóa 10 giây đầu tiên của tín hiệu thô **Wrist PPG (64 Hz)** và **Wrist ACC (32 Hz)** từ subject $S1$.

In [3]:
# ==============================================================================
# SECTION 4 — HIỂN THỊ TÍN HIỆU GỐC
# ==============================================================================
# Lấy 10 giây đầu tiên (640 mẫu PPG @ 64Hz, 320 mẫu ACC @ 32Hz)
t_ppg = np.arange(640) / 64.0
t_acc = np.arange(320) / 32.0

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

# Kênh PPG
axes[0].plot(t_ppg, ppg_raw[:640], color="#d62728", label="Wrist PPG (64 Hz)")
axes[0].set_ylabel("PPG (BVP)")
axes[0].set_title("Tin hieu tho ban dau cua Subject S1 (10 giay dau tien)")
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)

# 3 kênh Gia tốc kế (ACCx, ACCy, ACCz)
colors = ["#1f77b4", "#2ca02c", "#ff7f0e"]
labels = ["ACCx (32 Hz)", "ACCy (32 Hz)", "ACCz (32 Hz)"]
for i in range(3):
    axes[i + 1].plot(t_acc, acc_raw[:320, i], color=colors[i], label=labels[i])
    axes[i + 1].set_ylabel(f"ACC {chr(120+i)} (g)")
    axes[i + 1].legend(loc="upper right")
    axes[i + 1].grid(True, alpha=0.3)

axes[3].set_xlabel("Thoi gian (giay)")
plt.tight_layout()
plt.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_31436\1268637441.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Resample ACC 32 Hz $\to$ 64 Hz

Sử dụng bộ lọc Polyphase `scipy.signal.resample_poly` (thông qua `src.resample`) để nâng tần số lấy mẫu ACC từ 32 Hz lên 64 Hz, cân bằng độ dài với PPG.

In [4]:
# ==============================================================================
# SECTION 5 — RESAMPLE ACC 32 Hz -> 64 Hz
# ==============================================================================
# Nâng tần số lấy mẫu ACC lên 64 Hz
acc_resampled, _ = resample.resample_acc(acc_raw, fs_in=32.0, fs_out=64.0)

# Cắt aligned theo độ dài ngắn hơn giữa PPG và ACC
min_len = min(len(ppg_raw), len(acc_resampled))
ppg_aligned = ppg_raw[:min_len]
acc_aligned = acc_resampled[:min_len]

# Ghép matrix 4 kênh: [PPG, ACCx, ACCy, ACCz]
aligned_matrix = np.column_stack([ppg_aligned, acc_aligned])

print(f"[RESAMPLE] Shape ACC ban dau (32 Hz)   : {acc_raw.shape}")
print(f"[RESAMPLE] Shape ACC sau resample (64Hz): {acc_resampled.shape}")
print(f"[RESAMPLE] Shape PPG (64 Hz)           : {ppg_raw.shape}")
print(f"[RESAMPLE] Shape Aligned Matrix        : {aligned_matrix.shape} (N_samples, 4 channels)")

[RESAMPLE] Shape ACC ban dau (32 Hz)   : (294784, 3)
[RESAMPLE] Shape ACC sau resample (64Hz): (589568, 3)
[RESAMPLE] Shape PPG (64 Hz)           : (589568,)
[RESAMPLE] Shape Aligned Matrix        : (589568, 4) (N_samples, 4 channels)


---
## 6. Phân chia Fold và Thống kê Chuẩn hóa (Z-score)

Trong chiến lược 5-fold Subject-Wise CV:
- **Fold 1:** Train = $S6 \dots S15$, Val = $S4, S5$, Test = $S1, S2, S3$.
- **Quy tắc:** Thống kê Z-score (Mean & Std) chỉ được tính từ tập Train để tránh rò rỉ dữ liệu (data leakage).

In [5]:
# ==============================================================================
# SECTION 6 — FOLD VÀ NORMALIZATION
# ==============================================================================
import json

norm_path = PROJECT_ROOT / "configs" / "norm_stats_fold1.json"
with open(norm_path, "r", encoding="utf-8") as f:
    norm_stats_f1 = json.load(f)

# Bảng hiển thị thông số chuẩn hóa Fold 1
ch_names = norm_stats_f1.get("channels", norm_stats_f1.get("channel_names", ["PPG", "ACCx", "ACCy", "ACCz"]))
df_norm = pd.DataFrame({
    "Channel": ch_names,
    "Mean (Train Fold 1)": norm_stats_f1["mean"],
    "Std (Train Fold 1)": norm_stats_f1["std"],
    "DDOF": norm_stats_f1["ddof"]
})

print("[NORM] Bang Thong ke Z-score Fold 1 (Tinh CHI tu tap Train S6..S15):")
print(df_norm.to_string(index=False))

[NORM] Bang Thong ke Z-score Fold 1 (Tinh CHI tu tap Train S6..S15):
Channel  Mean (Train Fold 1)  Std (Train Fold 1)  DDOF
    PPG            -0.000341           86.878957     0
   ACCx            -0.543489            0.360015     0
   ACCy             0.096130            0.645617     0
   ACCz             0.311879            0.403271     0


---
## 7. Khởi tạo Cửa sổ Thử nghiệm Tập Test (Test Window)

Nạp tập dữ liệu Test đã xử lý của Fold 1 (`data/processed/fold1/test.npz`) và lấy deterministic cửa sổ hợp lệ đầu tiên của Subject $S1$.

In [6]:
# ==============================================================================
# SECTION 7 — TẠO TEST WINDOW
# ==============================================================================
test_npz_path = PROJECT_ROOT / "data" / "processed" / "fold1" / "test.npz"
if not test_npz_path.exists():
    raise FileNotFoundError(f"Khong tim thay file processed test data tai {test_npz_path}.")

test_data = np.load(test_npz_path, allow_pickle=True)
windows = test_data["windows"]            # Shape: (N, 4, 512)

if "metadata" in test_data:
    metadata_list = test_data["metadata"]
    subjs = np.array([m.get("subject", "S1") if isinstance(m, dict) else getattr(m, "subject", "S1") for m in metadata_list])
    window_ids = np.array([m.get("window_id", 0) if isinstance(m, dict) else getattr(m, "window_id", 0) for m in metadata_list])
    start_indices = np.array([m.get("start_index", 0) if isinstance(m, dict) else getattr(m, "start_index", 0) for m in metadata_list])
else:
    subjs = test_data.get("subjects", np.array(["S1"] * len(windows)))
    window_ids = test_data.get("window_ids", np.arange(len(windows)))
    start_indices = test_data.get("start_indices", np.zeros(len(windows), dtype=int))

# Chọn window đầu tiên của Subject S1
s1_mask = (subjs == "S1")
s1_indices = np.where(s1_mask)[0]
idx = s1_indices[0] if len(s1_indices) > 0 else 0

sample_window_norm = windows[idx]        # Shape: (4, 512) - Normalized
sample_subj = subjs[idx]
sample_win_id = window_ids[idx]
sample_start_idx = start_indices[idx]

print("[TEST] Test Window Duoc Chon (Deterministic):")
print(f"  - Subject       : {sample_subj}")
print(f"  - Window ID     : {sample_win_id}")
print(f"  - Start Index   : {sample_start_idx}")
print(f"  - Shape (C, T)  : {sample_window_norm.shape} (Expected: 4 channels, 512 samples)")

[TEST] Test Window Duoc Chon (Deterministic):
  - Subject       : S1
  - Window ID     : S1_test_f1_w0000
  - Start Index   : 0
  - Shape (C, T)  : (4, 512) (Expected: 4 channels, 512 samples)


---
## 8. Nén và Tái tạo bằng 1D-CNN Autoencoder

Nạp checkpoint thật **Fold 1, $d_b=8$, Seed 42** (`checkpoints/fold01_db08_seed42.pt`), thực hiện nén latent, đóng gói bitstream bằng `C1Codec` và giải nén tái tạo tín hiệu.

In [7]:
# ==============================================================================
# SECTION 8 — AUTOENCODER DEMO
# ==============================================================================
ckpt_path = PROJECT_ROOT / "checkpoints" / "fold01_db08_seed42.pt"
if not ckpt_path.exists():
    raise FileNotFoundError(f"Khong tim thay checkpoint {ckpt_path}.")

# Khởi tạo mô hình 1D-CNN Autoencoder d_b = 8
ae_model = model_module.C1Autoencoder(d_b=8).to(device)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)

if "model_state_dict" in checkpoint:
    ae_model.load_state_dict(checkpoint["model_state_dict"])
else:
    ae_model.load_state_dict(checkpoint)

ae_model.eval()

# Pipeline Nén & Tái tạo
input_tensor = torch.from_numpy(sample_window_norm).unsqueeze(0).float().to(device) # [1, 4, 512]

with torch.no_grad():
    latent_tensor = ae_model.encode(input_tensor) # [1, 8, 32]
    latent_np = latent_tensor.cpu().numpy().ravel() # M = 256 elements
    
    # 1. Đóng gói Binary Bitstream với 16-byte Header
    ae_bitstream = codec.encode_ae_bytes(latent_np, d_b=8)
    
    # 2. Giải mã Bitstream
    decoded_latent_np, meta_ae = codec.decode_ae_bytes(ae_bitstream)
    
    # 3. Qua Decoder 1D-CNN
    decoded_latent_tensor = torch.from_numpy(decoded_latent_np).view(1, 8, 32).float().to(device)
    rec_norm_ae = ae_model.decode(decoded_latent_tensor).cpu().numpy().squeeze(0) # [4, 512]

# Giải chuẩn hóa về đơn vị vật lý ban đầu
rec_phys_ae = normalize.denormalize(rec_norm_ae, norm_stats_f1)
orig_phys = normalize.denormalize(sample_window_norm, norm_stats_f1)

print("[AUTOENCODER] Ket qua Nen 1D-CNN Autoencoder (d_b=8, M=256):")
print(f"  - Input Normalized Shape : {sample_window_norm.shape}")
print(f"  - Latent Tensor Shape    : {tuple(latent_tensor.shape)}")
print(f"  - Latent Values (M)      : {len(latent_np)}")
print(f"  - Encoded Bitstream Size : {len(ae_bitstream)} Bytes (Header 16B + Payload {len(ae_bitstream)-16}B)")
print(f"  - Reconstructed Shape    : {rec_phys_ae.shape}")

[AUTOENCODER] Ket qua Nen 1D-CNN Autoencoder (d_b=8, M=256):


  - Input Normalized Shape : (4, 512)
  - Latent Tensor Shape    : (1, 8, 32)
  - Latent Values (M)      : 256
  - Encoded Bitstream Size : 1040 Bytes (Header 16B + Payload 1024B)
  - Reconstructed Shape    : (4, 512)


---
## 9. Nén và Tái tạo bằng DCT-II Top-K (Equal-Dimension)

Cấu hình $K = M = 256$ để so sánh cùng số lượng giá trị đại diện với Autoencoder.

In [8]:
# ==============================================================================
# SECTION 9 — DCT EQUAL-DIMENSION
# ==============================================================================
K_eq_dim = 256

# 1. DCT-II Top-K Encoding
topk_vals_dim, topk_inds_dim, ch_counts_dim, sparse_dct_encoded = baseline_dct.dct_encode_topk(
    sample_window_norm, k=K_eq_dim
)

# 2. Codec Bitstream Serialization
dct_bitstream_dim = codec.encode_dct_bytes(topk_vals_dim, topk_inds_dim, d_b=8, profile_id=1)

# 3. Codec Bitstream Deserialization
rec_topk_vals_dim, rec_topk_inds_dim, meta_dct_dim = codec.decode_dct_bytes(dct_bitstream_dim)

# 4. Reconstruct Sparse DCT Matrix and perform IDCT-II
sparse_dct_dim = np.zeros((4, 512), dtype=np.float32)
np.put(sparse_dct_dim, rec_topk_inds_dim, rec_topk_vals_dim)
rec_norm_dct_dim = baseline_dct.dct_decode(sparse_dct_dim)

# 5. Giải chuẩn hóa về đơn vị vật lý
rec_phys_dct_dim = normalize.denormalize(rec_norm_dct_dim, norm_stats_f1)

cr_dim_dct_dim = metrics.compute_cr_dim(2048, K_eq_dim)
cr_byte_64_dct_dim = metrics.RAW_BYTES_64HZ / len(dct_bitstream_dim)

print("[DCT-DIM] Ket qua Nen DCT Equal-Dimension (K=256):")
print(f"  - He so Top-K (K)        : {K_eq_dim}")
print(f"  - Encoded Bitstream Size : {len(dct_bitstream_dim)} Bytes")
print(f"  - Ty le nen chieu CR_dim  : {cr_dim_dct_dim:.1f}x")
print(f"  - Ty le nen byte CR_byte  : {cr_byte_64_dct_dim:.2f}x")

[DCT-DIM] Ket qua Nen DCT Equal-Dimension (K=256):
  - He so Top-K (K)        : 256
  - Encoded Bitstream Size : 1552 Bytes
  - Ty le nen chieu CR_dim  : 8.0x
  - Ty le nen byte CR_byte  : 5.28x


---
## 10. Nén và Tái tạo bằng DCT-II Top-K (Equal-Byte)

Cấu hình $K = \lfloor 4M / 6 \rfloor = 170$ và $4\text{ bytes padding}$ để khớp chính xác dung lượng bitstream thực tế $B_{\text{DCT}} = B_{\text{AE}} = 1040\text{ Bytes}$.

In [9]:
# ==============================================================================
# SECTION 10 — DCT EQUAL-BYTE
# ==============================================================================
eq_byte_budget = metrics.compute_equal_byte_budget(db=8)
K_eq_byte = eq_byte_budget.get("K_equal_byte", eq_byte_budget.get("K", 170))  # 170
padding_bytes = eq_byte_budget["padding"]  # 4 bytes

# 1. DCT-II Top-K Encoding
topk_vals_byte, topk_inds_byte, ch_counts_byte, sparse_dct_encoded_b = baseline_dct.dct_encode_topk(
    sample_window_norm, k=K_eq_byte
)

# 2. Codec Bitstream Serialization (Khớp Byte Budget)
dct_bitstream_byte = codec.encode_dct_bytes(
    topk_vals_byte, topk_inds_byte, d_b=8, profile_id=2, pad_bytes=padding_bytes
)

# 3. Codec Bitstream Deserialization
rec_topk_vals_byte, rec_topk_inds_byte, meta_dct_byte = codec.decode_dct_bytes(dct_bitstream_byte)

# 4. Reconstruct Sparse DCT Matrix and perform IDCT-II
sparse_dct_byte = np.zeros((4, 512), dtype=np.float32)
np.put(sparse_dct_byte, rec_topk_inds_byte, rec_topk_vals_byte)
rec_norm_dct_byte = baseline_dct.dct_decode(sparse_dct_byte)

# 5. Giải chuẩn hóa
rec_phys_dct_byte = normalize.denormalize(rec_norm_dct_byte, norm_stats_f1)

cr_dim_dct_byte = metrics.compute_cr_dim(2048, K_eq_byte)
cr_byte_64_dct_byte = metrics.RAW_BYTES_64HZ / len(dct_bitstream_byte)

print("[DCT-BYTE] Ket qua Nen DCT Equal-Byte (K=170):")
print(f"  - He so Top-K (K)        : {K_eq_byte}")
print(f"  - Padding Bytes          : {padding_bytes} Bytes")
print(f"  - Total Bitstream Size   : {len(dct_bitstream_byte)} Bytes (Khop 100% voi AE {len(ae_bitstream)}B)")
print(f"  - Ty le nen chieu CR_dim  : {cr_dim_dct_byte:.2f}x")
print(f"  - Ty le nen byte CR_byte  : {cr_byte_64_dct_byte:.2f}x")

[DCT-BYTE] Ket qua Nen DCT Equal-Byte (K=170):
  - He so Top-K (K)        : 170
  - Padding Bytes          : 4 Bytes
  - Total Bitstream Size   : 1040 Bytes (Khop 100% voi AE 1040B)
  - Ty le nen chieu CR_dim  : 12.05x
  - Ty le nen byte CR_byte  : 7.88x


---
## 11. Tính toán Chỉ số Méo dạng (Metrics: PRD, PRDN, RMSE)

Tính toán độc lập 3 chỉ số méo dạng cho từng kênh tín hiệu sinh học (**PPG, ACCx, ACCy, ACCz**) sử dụng công cụ `src.evaluate`. Tuân thủ quy tắc threshold mẫu số $threshold_c = N \cdot 10^{-12} \cdot \sigma_c^2$.

In [10]:
# ==============================================================================
# SECTION 11 — TÍNH METRICS
# ==============================================================================
metrics_rows = []
methods_list = [
    ("Autoencoder", "ae", rec_phys_ae, 8.0, metrics.RAW_BYTES_64HZ / len(ae_bitstream)),
    ("DCT Equal-Dim", "equal_dim", rec_phys_dct_dim, cr_dim_dct_dim, cr_byte_64_dct_dim),
    ("DCT Equal-Byte", "equal_byte", rec_phys_dct_byte, cr_dim_dct_byte, cr_byte_64_dct_byte)
]

channel_names = ["PPG", "ACCx", "ACCy", "ACCz"]
train_stds = norm_stats_f1["std"]

for method_name, comp_type, rec_phys, cr_dim_val, cr_byte_val in methods_list:
    for ch_idx, ch_name in enumerate(channel_names):
        x_ref = orig_phys[ch_idx]
        x_pred = rec_phys[ch_idx]
        sigma = train_stds[ch_idx]
        
        m = evaluate.compute_channel_metrics(x_ref, x_pred, channel_name=ch_name, sigma_train=sigma)
        
        metrics_rows.append({
            "channel": ch_name,
            "method": method_name,
            "comparison_type": comp_type,
            "CR_dim": round(cr_dim_val, 2),
            "CR_byte_64": round(cr_byte_val, 2),
            "PRD (%)": round(m["prd"], 2) if m["valid_prd"] else np.nan,
            "PRDN (%)": round(m["prdn"], 2) if m["valid_prdn"] else np.nan,
            "RMSE": round(m["rmse"], 4)
        })

df_metrics = pd.DataFrame(metrics_rows)
print("[METRICS] Bang Chi So Meo Dang Tai Tao (Test Window S1):")
print(df_metrics.to_string(index=False))

[METRICS] Bang Chi So Meo Dang Tai Tao (Test Window S1):
channel         method comparison_type  CR_dim  CR_byte_64  PRD (%)  PRDN (%)   RMSE
    PPG    Autoencoder              ae    8.00        7.88     6.89      6.89 4.2082
   ACCx    Autoencoder              ae    8.00        7.88     3.73     26.52 0.0318
   ACCy    Autoencoder              ae    8.00        7.88    11.57     17.71 0.0254
   ACCz    Autoencoder              ae    8.00        7.88     7.51     16.36 0.0388
    PPG  DCT Equal-Dim       equal_dim    8.00        5.28     4.18      4.18 2.5528
   ACCx  DCT Equal-Dim       equal_dim    8.00        5.28     2.16     15.35 0.0184
   ACCy  DCT Equal-Dim       equal_dim    8.00        5.28     9.57     14.65 0.0210
   ACCz  DCT Equal-Dim       equal_dim    8.00        5.28     3.73      8.12 0.0192
    PPG DCT Equal-Byte      equal_byte   12.05        7.88     6.24      6.24 3.8122
   ACCx DCT Equal-Byte      equal_byte   12.05        7.88     3.25     23.07 0.0277
   ACCy 

---
## 12. Trực quan hóa So sánh Dạng sóng Tái tạo (Original vs. Reconstructed)

Vẽ đồ thị so sánh dạng sóng vật lý gốc với 3 phương pháp nén trên 2 kênh đại diện: **PPG** và **ACCx**.

In [11]:
# ==============================================================================
# SECTION 12 — VẼ ORIGINAL VS RECONSTRUCTION
# ==============================================================================
t_win = np.arange(512) / 64.0  # 8 giây

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Kênh PPG
axes[0].plot(t_win, orig_phys[0], color="black", linewidth=2.0, label="Original PPG")
axes[0].plot(t_win, rec_phys_ae[0], color="#d62728", linestyle="--", linewidth=1.5, label="AE (d_b=8)")
axes[0].plot(t_win, rec_phys_dct_dim[0], color="#1f77b4", linestyle=":", linewidth=1.5, label="DCT Equal-Dim (K=256)")
axes[0].plot(t_win, rec_phys_dct_byte[0], color="#2ca02c", linestyle="-.", linewidth=1.5, label="DCT Equal-Byte (K=170)")
axes[0].set_ylabel("PPG (BVP)")
axes[0].set_title(f"So sanh Dang song Tai tao — Subject {sample_subj}, Window #{sample_win_id}")
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)

# Kênh ACCx
axes[1].plot(t_win, orig_phys[1], color="black", linewidth=2.0, label="Original ACCx")
axes[1].plot(t_win, rec_phys_ae[1], color="#d62728", linestyle="--", linewidth=1.5, label="AE (d_b=8)")
axes[1].plot(t_win, rec_phys_dct_dim[1], color="#1f77b4", linestyle=":", linewidth=1.5, label="DCT Equal-Dim (K=256)")
axes[1].plot(t_win, rec_phys_dct_byte[1], color="#2ca02c", linestyle="-.", linewidth=1.5, label="DCT Equal-Byte (K=170)")
axes[1].set_ylabel("ACCx (g)")
axes[1].set_xlabel("Thoi gian trong cua so (giay)")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_31436\1223653218.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 13. Nạp Kết quả Thử nghiệm Tổng thể (Final Experiment Results)

Nạp tập dữ liệu kết quả thực nghiệm thật từ file `results/results.csv` (hoặc `results/results.csv.gz`).

In [12]:
# ==============================================================================
# SECTION 13 — LOAD FINAL EXPERIMENT RESULTS
# ==============================================================================
results_csv_path = PROJECT_ROOT / "results" / "results.csv"
results_gz_path = PROJECT_ROOT / "results" / "results.csv.gz"

results_df = None
if results_csv_path.exists():
    results_df = pd.read_csv(results_csv_path)
elif results_gz_path.exists():
    results_df = pd.read_csv(results_gz_path, compression="gzip")

if results_df is not None and not results_df.empty:
    print(f"[RESULTS] Da nap thanh cong file ket qua thuc nghiem that: {len(results_df):,} dong.")
    print(f"  - Danh sach Subjects ({results_df['subject'].nunique()}): {sorted(results_df['subject'].unique())}")
    print(f"  - Danh sach Folds ({results_df['fold'].nunique()}): {sorted(results_df['fold'].unique())}")
    print(f"  - Phuong phap ({results_df['method'].nunique()}): {results_df['method'].unique().tolist()}")
    print(f"  - Cac loai so sanh: {results_df['comparison_type'].unique().tolist()}")
    print(f"  - Cac muc nen d_b: {sorted(results_df['db'].dropna().unique())}")
else:
    print("[WARNING] Final results.csv chua san sang. Hay chay src.evaluate_all truooc.")

[RESULTS] Da nap thanh cong file ket qua thuc nghiem that: 1,035,584 dong.
  - Danh sach Subjects (15): ['S1', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
  - Danh sach Folds (5): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  - Phuong phap (2): ['AE', 'DCT']
  - Cac loai so sanh: ['equal_dim', 'equal_byte']
  - Cac muc nen d_b: [np.int64(2), np.int64(4), np.int64(8), np.int64(16)]


---
## 14. Hiển thị Bảng Tổng hợp Báo cáo (Overall Summaries)

Hiển thị thông số trung bình theo mức nén từ các file summary kết quả thực nghiệm.

In [13]:
# ==============================================================================
# SECTION 14 — HIỂN THỊ SUMMARY
# ==============================================================================
summary_dim_path = PROJECT_ROOT / "results" / "overall_summary_equal_dim.csv"
summary_byte_path = PROJECT_ROOT / "results" / "overall_summary_equal_byte.csv"

if summary_dim_path.exists():
    df_sum_dim = pd.read_csv(summary_dim_path)
    print("[SUMMARY] Bang Tong hop Ket qua (Equal-Dimension):")
    print(df_sum_dim.head(10).to_string(index=False))

if summary_byte_path.exists():
    df_sum_byte = pd.read_csv(summary_byte_path)
    print("\n[SUMMARY] Bang Tong hop Ket qua (Equal-Byte):")
    print(df_sum_byte.head(10).to_string(index=False))

[SUMMARY] Bang Tong hop Ket qua (Equal-Dimension):
method comparison_type  db  budget channel  n_subjects  prd_mean  prd_median   prd_p90  invalid_rate_prd  prdn_mean  prdn_median   prdn_p90  invalid_rate_prdn  rmse_mean  rmse_median  rmse_p90
    AE       equal_dim  16      16     PPG          15  8.126209    6.821110 12.671558          0.000000   8.135814     6.826382  12.681910           0.000000   4.894261     4.079344  8.995376
    AE       equal_dim  16      16    ACCx          15  5.297213    4.084380  9.680502          0.000000  59.116893    15.119728  73.688197           0.000000   0.025076     0.018621  0.046294
    AE       equal_dim  16      16    ACCy          15 15.458249   10.061279 32.262017          0.001053 437.908175    41.681839 136.603000           0.001053   0.066448     0.046809  0.126124
    AE       equal_dim  16      16    ACCz          15 10.147671    8.016589 20.514868          0.000000  61.668555    23.747348  93.639698           0.000000   0.042146     0.0

---
## 15. Đồ thị Đường cong Tỷ lệ Nén vs. Độ Méo Dạng (CR–PRD Curves)

Vẽ đồ thị so sánh mối quan hệ giữa Tỷ lệ nén chiều ($CR_{\text{dim}}$) / Tỷ lệ nén byte ($CR_{\text{byte}}$) và độ méo dạng PRD (%) trên tập Test thật.

In [14]:
# ==============================================================================
# SECTION 15 — VẼ CR–PRD TỪ KẾT QUẢ THẬT
# ==============================================================================
if results_df is not None and not results_df.empty:
    ppg_df = results_df[results_df["channel"] == "PPG"].dropna(subset=["PRD"])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Đồ thị A: CR_dim vs PRD
    for method, grp in ppg_df.groupby("method"):
        agg = grp.groupby("CR_dim")["PRD"].mean().reset_index()
        axes[0].plot(agg["CR_dim"], agg["PRD"], marker="o", label=f"Method: {method}")
    axes[0].set_xlabel("Ty le nen chieu (CR_dim)")
    axes[0].set_ylabel("PRD Trung binh (%)")
    axes[0].set_title("A. Ty le Nen Chieu vs Meo Dang PRD (Kenh PPG)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # Đồ thị B: CR_byte_64 vs PRD
    for method, grp in ppg_df.groupby("method"):
        agg = grp.groupby("CR_byte_64")["PRD"].mean().reset_index()
        axes[1].plot(agg["CR_byte_64"], agg["PRD"], marker="s", label=f"Method: {method}")
    axes[1].set_xlabel("Ty le nen byte (CR_byte_64)")
    axes[1].set_ylabel("PRD Trung binh (%)")
    axes[1].set_title("B. Ty le Nen Byte vs Meo Dang PRD (Kenh PPG)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("[WARNING] Khong co du lieu results.csv de ve do thi CR-PRD.")

C:\Users\PC\AppData\Local\Temp\ipykernel_31436\465895235.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 16. So sánh Cặp Paired Comparison ($\delta = \text{AE} - \text{DCT}$)

Hiển thị bảng so sánh sai biệt trực tiếp $\delta_s = \text{AE} - \text{DCT}$ trên cùng đối tượng/cửa sổ.

In [15]:
# ==============================================================================
# SECTION 16 — PAIRED COMPARISON
# ==============================================================================
paired_dim_path = PROJECT_ROOT / "results" / "paired_comparison_equal_dim.csv"

if paired_dim_path.exists():
    df_paired = pd.read_csv(paired_dim_path)
    print("[PAIRED] Bang So Sanh Cap Paired Comparison (Equal-Dimension):")
    print(df_paired.head(10).to_string(index=False))
    print("\n[INFO] Ghi chu giaichich:")
    print("  - delta = AE - DCT (tinh theo tung subject/channel/window_id)")
    print("  - delta < 0: AE co do meo dang (PRD/RMSE) thap hon DCT.")
    print("  - delta > 0: DCT co do meo dang thap hon AE.")
else:
    print("[WARNING] Chua co file paired_comparison_equal_dim.csv.")

[PAIRED] Bang So Sanh Cap Paired Comparison (Equal-Dimension):
subject comparison_type  db  budget channel  prd_mean_ae  prd_mean_dct  delta_prd_mean  ae_wins_prd  prdn_mean_ae  prdn_mean_dct  delta_prdn_mean  ae_wins_prdn  rmse_mean_ae  rmse_mean_dct  delta_rmse_mean  ae_wins_rmse  delta_prd_median  delta_prdn_median  delta_rmse_median
     S1       equal_dim  16      16     PPG     6.150509      2.046874        4.103636        False      6.157627       2.049204         4.108423         False      5.251236       1.768326         3.482910         False          3.750375           3.758313           3.032383
     S1       equal_dim  16      16    ACCx     5.042240      1.858156        3.184083        False     77.822852       7.809132        70.013720         False      0.026280       0.010048         0.016232         False          2.673325           9.659243           0.012288
     S1       equal_dim  16      16    ACCy    15.385525      5.530655        9.854871        False   1559.80

---
## 17. Kết luận Notebook

- **Dữ liệu thực tế:** Toàn bộ minh họa trong Notebook được thực hiện 100% trên dữ liệu thực tế PPG-DaLiA ($S1 \dots S15$).
- **Luồng xử lý hoàn chỉnh:** Kiểm chứng thành công luồng xử lý từ Preprocessing $\to$ Resampling $\to$ Normalization $\to$ Bottleneck Compression (1D-CNN AE & DCT-II) $\to$ Bitstream Codec $\to$ Signal Reconstruction & Metric Evaluation.
- **Khớp ngân sách đa chiều:** Hệ thống đã thực thi thành công cả 2 chế độ so sánh: **Equal-Dimension** ($K=M$) và **Equal-Byte** ($B_{\text{DCT}} = B_{\text{AE}}$).
- **Phân định rõ ràng:** Notebook này phục vụ mục đích trình diễn và minh họa trực quan. Các kết quả thực nghiệm tổng thể 20 main runs và 15 seed runs được duy trì nhất quán từ các script tự động trong thư mục `src/`.